In [5]:
!curl --data '{"email": "youxitiankaggle@gmail.com", "description": "Quant Research"}' https://fraser.stlouisfed.org/api/api_key

{"message":"API key created and sent via email."}

In [12]:
import os
import time
from pathlib import Path

import requests
from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display

# 1. 初期設定と認証（ヘッダーの設定）
API_KEY = "6da0308c620b2308091cb2c902a0f5db"  # 取得したAPIキーを入力
HEADERS = {"X-API-Key": API_KEY}
BASE_URL = "https://fraser.stlouisfed.org/api"

PRJ_DIR = Path().cwd()
DATA_DIR = PRJ_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

In [21]:
# 2. FOMC包括タイトル(ID: 1262)に紐づく全ての会合アイテムを取得
title_id = 677
items_url = f"{BASE_URL}/title/{title_id}/items"

print("FOMCのアイテムリストを取得中...")
response = requests.get(items_url, headers=HEADERS)
response.raise_for_status()
items_data = response.json()

display(items_data)

FOMCのアイテムリストを取得中...


{'format': 'json',
 'page': 1,
 'limit': 100,
 'fields': ['*'],
 'total': 1038,
 'records': [{'language': ['eng'],
   'location': {'url': ['https://fraser.stlouisfed.org/title/federal-open-market-committee-meeting-minutes-transcripts-documents-677/meeting-july-20-1933-22634'],
    'iiif_manifest_url': ['https://iiif.slf.digirati.io/presentation/item/22634'],
    'apiUrl': ['https://fraser.stlouisfed.org/api//22634'],
    'pdfUrl': ['https://fraser.stlouisfed.org/docs/historical/FOMC/meetingdocuments/rg82_fomcminutes_19330720.pdf'],
    'textUrl': ['https://fraser.stlouisfed.org/files/text/historical/FOMC/meetingdocuments/rg82_fomcminutes_19330720.txt']},
   'titleInfo': [{'title': 'Meeting, July 20, 1933',
     'subTitle': '',
     'titlePartNumber': ''}],
   'originInfo': {'issuance': 'periodical',
    'sortDate': '1933-07-20',
    'dateOther': '',
    'frequency': '',
    'dateIssued': ['July 20, 1933']},
   'recordInfo': {'recordIdentifier': [22634],
    'recordUpdatedDate': '2025-0

In [8]:
# 3. 各アイテムをループ処理し、Minutesを特定してテキストをダウンロード
for item in items_data.get("items", []):
    item_id = item.get("item_id")
    item_title = item.get(
        "title", ""
    )  # 文書名（例: "Minutes of the Federal Open Market Committee"）
    item_date = item.get("date", "unknown_date")  # 開催日 (YYYY-MM-DD)

    # テキストマイニングの対象を「Minutes（議事録）」に絞り込む（StatementやTranscriptsを除外）
    if "Minutes" in item_title:
        print(f"対象を発見: {item_date} - {item_title} (Item ID: {item_id})")

        # 4. 個別のItem IDから、添付されているファイル情報（PDF/TXT等）を取得
        files_url = f"{BASE_URL}/item/{item_id}/files"
        files_response = requests.get(files_url, headers=HEADERS)

        if files_response.status_code == 200:
            files_data = files_response.json()

            # ファイルリストの中からプレインテキスト(.txt)のURLを探索
            for file_info in files_data.get("files", []):
                file_url = file_info.get("download_url")

                if file_url and file_url.endswith(".txt"):
                    print(f" -> テキストURLを特定: {file_url}")

                    # 5. テキストデータのダウンロードと保存
                    txt_response = requests.get(file_url, headers=HEADERS)
                    if txt_response.status_code == 200:
                        # ファイル名を設定 (例: 2018-03-21_minutes.txt)
                        file_name = f"{item_date}_minutes.txt"
                        file_path = DATA_DIR / file_name

                        with open(file_path, "w", encoding="utf-8") as f:
                            f.write(txt_response.text)

                        print(f" -> 保存完了: {file_path}")
                    break

        # FRASERサーバーへの負荷軽減とAPI制限回避のためのインターバル
        time.sleep(1)

print("すべてのMinutesのテキスト取得が完了しました。")

すべてのMinutesのテキスト取得が完了しました。
